In [2]:
import sys
from pathlib import Path

import pandas as pd


def find_project_root(start: Path) -> Path:
    start = start.resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate

    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [3]:
from src.loader import load_candidates
from src.preprocessing import prepare_data
from src.feature_engineering import build_all_features
from src.ranking import *

In [4]:
candidates = load_candidates(DATA_DIR / "candidates.jsonl")

data = prepare_data(candidates)

features = build_all_features(
    profiles_df=data["profiles"],
    career_df=data["career"],
    skill_df=data["skills"],
    signals_df=data["signals"]
)

In [5]:
semantic_scores = pd.read_parquet(DATA_DIR / "semantic_scores.parquet")

behavior_scores = pd.read_parquet(DATA_DIR / "behavior_scores.parquet")

In [6]:
ranking_df = (
    features
    .merge(semantic_scores, on="candidate_id")
    .merge(behavior_scores, on="candidate_id")
)

technical = build_technical_score(features)

product = build_product_fit_score(features)

experience = build_experience_fit_score(features)

ranking_df = build_final_score(
    ranking_df,
    technical,
    product,
    experience
)

ranking_df = rank_candidates(ranking_df)

In [7]:
top100 = ranking_df.head(100)


top100.head()

,candidate_id,years_of_experience,current_title,total_jobs,unique_companies,avg_job_duration,shortest_job,longest_job,product_company_ratio,service_company_ratio,...,semantic_similarity,availability_score,recruiter_interest_score,trust_score,engagement_score,behavior_score,technical_score,product_fit_score,experience_fit_score,final_score
0,CAND_0077337,7.0,Staff Machine Learning Engineer,4,4,20.750000,6,44,0.5,0.0,...,0.782215,0.907800,0.292314,0.8584,0.6525,0.851117,0.839801,0.804665,1.000000,0.833849
1,CAND_0046064,8.9,Senior NLP Engineer,3,3,35.333333,34,36,0.0,0.0,...,0.877985,0.876148,0.640024,0.9012,0.4446,0.912841,0.833262,0.521866,0.857143,0.832100
2,CAND_0081846,6.7,Lead AI Engineer,2,2,39.500000,27,52,1.0,0.0,...,0.849290,0.861580,0.508979,0.9828,0.4864,0.893847,0.654356,1.000000,1.000000,0.827635
3,CAND_0052682,6.6,NLP Engineer,2,2,39.000000,32,46,0.0,0.0,...,0.752276,0.910898,0.542406,0.9416,0.7678,1.000000,0.864248,0.451895,1.000000,0.817761
4,CAND_0002025,5.9,Senior AI Engineer,2,2,35.000000,28,42,0.0,0.0,...,0.949568,0.896470,0.381545,0.9272,0.6153,0.886659,0.706972,0.539359,0.857143,0.817090


In [8]:
print("Total Candidates :", len(top100))

print("Unique IDs :", top100["candidate_id"].nunique())

print("Missing Values :")

print(top100.isna().sum())

Total Candidates : 100
Unique IDs : 100
Missing Values :
candidate_id            0
years_of_experience     0
current_title           0
total_jobs              0
unique_companies        0
                       ..
behavior_score          0
technical_score         0
product_fit_score       0
experience_fit_score    0
final_score             0
Length: 67, dtype: int64


In [9]:
top100 = top100.copy()

top100["rank"] = range(1, 101)

In [10]:
# `top100` already contains these feature columns from `ranking_df`.
# Avoid re-merging them here, because duplicate column names would become
# suffixed (`_x` / `_y`) and break the reasoning cell below.
top100 = top100.copy()

In [11]:
def generate_reasoning(row):

    reasons = []

    # Experience + title
    reasons.append(
        f"{row['years_of_experience']:.1f} years as {row['current_title']}"
    )

    # JD fit
    if row["semantic_similarity"] >= 0.85:
        reasons.append(
            "excellent semantic match to the AI Engineer job description"
        )
    elif row["semantic_similarity"] >= 0.75:
        reasons.append(
            "strong semantic match to the required profile"
        )

    # Retrieval / Ranking
    if row["retrieval_mentions"] > 0:
        reasons.append(
            "demonstrates retrieval or ranking experience"
        )

    # Embeddings
    if row["embedding_skill_count"] > 0:
        reasons.append(
            "has embeddings experience"
        )

    # Vector databases
    if row["vector_db_skill_count"] > 0:
        reasons.append(
            "worked with vector databases"
        )

    # Product companies
    if row["product_company_ratio"] >= 0.50:
        reasons.append(
            "strong product-company background"
        )

    elif row["service_company_ratio"] >= 0.80:
        reasons.append(
            "primarily service-company experience"
        )

    # Technical strength
    if row["technical_score"] >= 0.80:
        reasons.append(
            "excellent technical strength"
        )

    elif row["technical_score"] >= 0.65:
        reasons.append(
            "solid technical profile"
        )

    # Behaviour
    if row["behavior_score"] >= 0.80:
        reasons.append(
            "high recruiter engagement and availability"
        )

    elif row["behavior_score"] >= 0.60:
        reasons.append(
            "good recruiter engagement"
        )

    # Honest concern
    if row["years_of_experience"] < 5:
        reasons.append(
            "slightly below the preferred experience range"
        )

    return ". ".join(reasons) + "."

In [12]:
top100["reasoning"] = top100.apply(
    generate_reasoning,
    axis=1
)

In [13]:
submission = top100[
    [
        "candidate_id",
        "rank",
        "final_score",
        "reasoning"
    ]
].copy()

submission.rename(
    columns={
        "final_score": "score"
    },
    inplace=True
)

In [14]:
assert len(submission) == 100

assert submission["candidate_id"].nunique() == 100

assert submission["rank"].tolist() == list(range(1, 101))

assert submission["score"].is_monotonic_decreasing

print("Submission validated successfully!")

Submission validated successfully!


In [15]:
TEAM_ID = "YOUR_TEAM_ID"
output_path = DATA_DIR / f"{TEAM_ID}.csv"

submission.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print(f"Saved {output_path}")

Saved C:\Users\vishal\Desktop\Redrobe-Ranker\data\YOUR_TEAM_ID.csv


In [16]:
submission.head(10)

,candidate_id,rank,score,reasoning
0,CAND_0077337,1,0.833849,7.0 years as Staff Machine Learning Engineer. ...
1,CAND_0046064,2,0.832100,8.9 years as Senior NLP Engineer. excellent se...
2,CAND_0081846,3,0.827635,6.7 years as Lead AI Engineer. strong semantic...
3,CAND_0052682,4,0.817761,6.6 years as NLP Engineer. strong semantic mat...
4,CAND_0002025,5,0.817090,5.9 years as Senior AI Engineer. excellent sem...
5,CAND_0008425,6,0.809577,7.8 years as Senior NLP Engineer. strong seman...
6,CAND_0083307,7,0.798478,7.8 years as Search Engineer. demonstrates ret...
7,CAND_0088025,8,0.790886,8.6 years as Staff Machine Learning Engineer. ...
8,CAND_0007009,9,0.787961,7.9 years as Recommendation Systems Engineer. ...
9,CAND_0081686,10,0.782862,6.0 years as Search Engineer. strong semantic ...
